# Medición CUDA en GPU gratuita
La libreta se ejecuta **solamente cuando una GPU Colab está conectada**. Los números son latencias en la GPU asignada por Colab, no en la GPU oficial Blackwell. Usa el oráculo independiente y se detiene ante fallos de exactitud. No publica ninguna candidatura ni realiza pagos.
**Instrucciones:** en Entorno de ejecución > Cambiar tipo de entorno de ejecución elige T4 GPU (si aparece); después usa Ejecutar todas.


In [ ]:
import subprocess, re
r = subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"], capture_output=True, text=True)
print(r.stdout or r.stderr)
if r.returncode: raise RuntimeError("No hay GPU asignada. Selecciona GPU en Entorno de ejecución.")
m = re.search(r"(\d+)\.(\d+)\s*$", r.stdout.strip().splitlines()[0])
if not m: raise RuntimeError("No pude detectar compute_cap")
ARCH = "sm_" + m.group(1) + m.group(2)
print("Architecture:", ARCH)
print(subprocess.run(["nvcc", "--version"],capture_output=True,text=True).stdout[-400:])

In [ ]:
import subprocess, pathlib, urllib.request
root=pathlib.Path("/content/fherma-test")
if not (root/".git").exists(): subprocess.run(["git","clone","--depth","1","--branch","research-fused-reindex-no-profiler","https://github.com/aritomas3123-boop/fherma-gpu-polymul.git",str(root)],check=True)
oracle=root/"oracle.py"
if not oracle.exists(): urllib.request.urlretrieve("https://raw.githubusercontent.com/fherma-ai/polynomial-mult-cupqc-example/main/oracle.py",oracle)
def run(*args):
 print("+", " ".join(map(str,args)))
 subprocess.run([str(a) for a in args],check=True)
run("cmake","-S",root,"-B","/content/fherma-baseline","-DCMAKE_BUILD_TYPE=Release",f"-DFHERMA_CUDA_ARCH={ARCH}","-DFHERMA_FUSED_REINDEX=OFF","-DFHERMA_PROFILE=OFF")
run("cmake","--build","/content/fherma-baseline","-j","2")
run("cmake","-S",root,"-B","/content/fherma-fused","-DCMAKE_BUILD_TYPE=Release",f"-DFHERMA_CUDA_ARCH={ARCH}","-DFHERMA_FUSED_REINDEX=ON","-DFHERMA_PROFILE=OFF")
run("cmake","--build","/content/fherma-fused","-j","2")

In [ ]:
import shutil, json, pathlib, sys
for N,W in [(128,128),(1024,128),(128,1747)]:
 case=pathlib.Path(f"/content/fherma-case-{N}-{W}")
 if case.exists(): shutil.rmtree(case)
 run(sys.executable,oracle,"make",case,N,W,2)
 for label in ["baseline","fused"]:
  shutil.rmtree(case/"out",ignore_errors=True)
  run(f"/content/fherma-{label}/solution",case)
  run(sys.executable,oracle,"verify",case)
  results=json.loads((case/"out"/"results.json").read_text())
  shutil.copyfile(case/"out"/"results.json",f"/content/fherma-{label}-results-{N}-{W}.json")
  print("RESULT",N,W,label,results)
print("Las pruebas anteriores NO son medidas de la GPU oficial de FHERMA.")

In [ ]:
import json, pathlib, statistics
print("RESUMEN DE LATENCIAS: MILISEGUNDOS. Solo GPU Colab; no comparar con el ranking oficial.")
for N,W in [(128,128),(1024,128),(128,1747)]:
    vals={}
    for name in ["baseline","fused"]:
        p=pathlib.Path(f"/content/fherma-{name}-results-{N}-{W}.json")
        if p.exists():
            vals[name]=statistics.median([c["seconds"]*1000 for c in json.loads(p.read_text())["cases"] if c["status"]=="ok"])
    print(N,W,vals)

## Prueba de tamaños reales de FHERMA (GPU de Colab)
Esta prueba usa 32.768×868 y 65.536×1.747, con un operando aleatorio completo y otro con dos coeficientes no nulos para verificar la salida exactamente. Se descarta el primer caso antes de calcular la mediana. **No sustituye el benchmark oficial**, pero permite comparar la ejecución completa de las dos variantes en T4.


In [ ]:
import subprocess
subprocess.run(['python3','/content/fherma-test/benchmark_large.py'],check=True)